# Mini Project 1 — Analysis Notebook

**Your name:**  May Ng
**Dataset:**  Ravelry Patterns in Jul 2025 - Jun 2026
**Date:**  5/12/2026

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [3]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")

Setup complete.


---

## Section 1 — Overview

**Dataset:** *Ravelry's recently popular pattern data for Jul 2025 - Jun 2026*

**Why this dataset:** *I chose Ravelry because its pattern popularity and seasonality data mirrors real-world retail business insights for designers, and as a crafter myself, it's a domain I find genuinely compelling.*

**Three analytical questions:**

1. *What type (hat, socks, t-shirt, sweater, etc.) of project is most popular by season?*
2. *What weight and fiber of yarn is most popular by season? *
3. *Which designers are the most popular and how much experience (based on first Ravelry publishing date) do they have?*

**What a practitioner would do with these findings:** *A practitioner would use these findings to advise designers and/or yarn brands and dyers of the trends to guide their design and product offering plans. *

In [4]:
# Load your dataset
# Replace 'your_dataset.csv' with your actual filename.
# The file should be in the same folder as this notebook.
# If you're loading from an API result, replace pd.read_csv() with the appropriate call.
#
# Example (app review dataset from class):
# df = pd.read_csv('app_reviews_demo.csv')

df = pd.read_csv('data/ravelry_patterns_2026.csv')  # ← replace with your filename

print(df.shape)
df.head()

(7332, 21)


,month_collected,season,craft,pattern_id,pattern_name,published_date,category,yarn_weight,fiber,projects_count,...,queued_projects_count,rating_average,rating_count,designer_id,designer_name,designer_first_published,designer_years_experience,designer_total_projects,designer_total_favorites,designer_pattern_count
0,7,Summer,knitting,7480818,CPH Sweater,2025-11-01,Pullover,Worsted,Alpaca,135.0,...,1595.0,4.680000,25.0,145180.0,"lesfillesducoeur, Lilly Schwarzkopf",2025-11-01,1.2,135.0,15697.0,1.0
1,7,Summer,knitting,7478239,Harvest Flower Sweater,2025-11-01,Pullover,DK,Rambouillet,386.0,...,2127.0,4.490196,51.0,105302.0,Jessie Maed Designs,2025-11-01,1.2,539.0,30773.0,2.0
2,7,Summer,knitting,1133877,Old Fashioned,2021-04-01,Pullover,Worsted,Wool,103.0,...,856.0,4.857143,35.0,5853.0,Thea Colman,2021-04-01,5.7,193.0,16389.0,2.0
3,7,Summer,knitting,7495801,Melt the ICE Hat,2026-01-01,"Beanie, Toque",Any gauge - designed for any gauge,Wool,13298.0,...,4262.0,4.673605,2402.0,73221.0,Paul S Neary,2026-01-01,1.0,13298.0,35902.0,1.0
4,7,Summer,knitting,162969,Wolf Cardigan,NaN,Cardigan,Bulky,Wool,462.0,...,550.0,4.350000,40.0,3471.0,Mary Maxim,NaN,NaN,462.0,4354.0,1.0


---

## Section 2 — Data Profile 

**Questions**
- *How many rows and columns does your dataset have?* 7332 rows x 21 columns.
- *What does each column represent?* A property of a pattern in Ravelry's database. Columns such as season and the designer properties are calculated in the fetch request based on the type of data we needed for our insights.

The 21 columns fall into three groups based on how the fetch script (`fetch_ravelry_data.py`) produces them:

**Calculated by the fetch script (6)** — these don't exist in the API:
- `season` — mapped from `month_collected` (Dec–Feb = Winter, Mar–May = Spring, etc.) (This later proved to be faulty data)
- `designer_first_published` — earliest `published_date` across that designer's patterns *in this dataset*
- `designer_years_experience` — years from `designer_first_published` to 2026-12-31, rounded to 1 decimal
- `designer_total_projects` / `designer_total_favorites` — sums of `projects_count` / `favorites_count` over the designer's patterns in the dataset
- `designer_pattern_count` — count of the designer's unique patterns in the dataset

*Limitation: the designer stats are computed only over patterns that appear in this sample, not the designer's full Ravelry catalog. While this is not representative of the true designer's stats, the fetch limitations of the free API and size of data collection is beyond the scope of this project. This dataset and analysis is used to demonstrate skills and learning for a course.*

**Derived/transformed from API fields (5):**
- `month_collected` — the `started=YYYY-MM` month used in the project search query, not a field on any record (Did not properly return unique results as expected)
- `fiber` — highest-percentage fiber category from a separate `/yarns.json` lookup on the pattern's first yarn pack
- `yarn_weight` — parsed from `yarn_weight_description` (e.g. "Worsted (9 wpi)" → "Worsted")
- `category` — first entry of the pattern's `pattern_categories` list
- `published_date` — reformatted from `YYYY/MM/DD` to `YYYY-MM-DD`

**Direct from the API (10):** `craft` (search query parameter), `pattern_id`, `pattern_name`, `projects_count`, `favorites_count`, `queued_projects_count`, `rating_average`, `rating_count`, `designer_id`, `designer_name` — taken as-is, with missing values filled (0 for counts, "Unknown" for text).

- *Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?* There are missing pattern published dates, which can affect the designer columns that rely on them, such as `designer_first_published` and `designer_years_experience`.
- *Which column or columns will your analysis focus on, and why?* `category`, `season`, `fiber`, and `yarn_weight`, as those are the columns relevant to the questions I want to answer for my hypothetical scenario of informing designers/fiber vendors of trends in the industry.

In [5]:
# Check column types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7332 entries, 0 to 7331
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   month_collected            7332 non-null   int64  
 1   season                     7332 non-null   str    
 2   craft                      7332 non-null   str    
 3   pattern_id                 7332 non-null   int64  
 4   pattern_name               7332 non-null   str    
 5   published_date             6744 non-null   str    
 6   category                   7332 non-null   str    
 7   yarn_weight                7332 non-null   str    
 8   fiber                      7332 non-null   str    
 9   projects_count             7320 non-null   float64
 10  favorites_count            7320 non-null   float64
 11  queued_projects_count      7320 non-null   float64
 12  rating_average             7320 non-null   float64
 13  rating_count               7320 non-null   float64
 14  des

In [ ]:
# Count of patterns by season
pattern_counts = (
    df[df["season"] != "Unknown"]["season"]
      .str.strip()
      .value_counts()
      .head(4)
      .reset_index()
)
pattern_counts.columns = ["season", "count"]

fig = px.bar(
    pattern_counts,
    x="season",
    y="count",
    title="Season Popularity by Pattern Count",
    labels={"season": "Season", "count": "Appearances (pattern-months)"}
)
fig.show()

pattern_counts

,season,count
0,Summer,1833
1,Fall,1833
2,Winter,1833
3,Spring,1833


**Data quality issues I noticed:**

- **Every season has exactly 1,833 rows.** It turns out my data originally called the same exact set of data for all 12 months! The AI agent kept hallucinating and creating solutions that didn't work. This exposed that that users rarely log when they start their projects, which was a basis for how the fetch was created. 
- **One pattern has no details at all.** The Ravelry batch API returns 404 for an entire batch if it includes a deleted pattern; the fetch script splits failed batches in half and retries, so only a single deleted pattern (12 rows, "Unknown" category) had to be dropped. I exclude those rows from category/fiber/weight analyses.
- **Fiber is the weakest field** (1,152 rows Unknown, ~16%): fiber isn't part of the pattern record on Ravelry, so it's looked up from each pattern's first listed yarn — patterns with no linked yarn (e.g. handspun) can't be resolved, and some yarns have no fiber breakdown. Yarn weight is Unknown for ~6% of rows for similar reasons.
- **A few extreme values:** published dates go back to 1945 (vintage patterns republished on Ravelry), which produces designer "experience" values up to 82 years. The experience figure is also a proxy — it's based only on patterns captured in *this* dataset, so it can understate a designer's real history.

**Columns my analysis will focus on:**  `category` (Q1), `yarn_weight` and `fiber` (Q2), and the designer columns — `designer_name`, `designer_total_projects`, `designer_total_favorites`, `designer_years_experience` (Q3). The popularity counts (`projects_count`, `favorites_count`) let me weight patterns by how much they're actually used rather than treating all patterns equally.

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1:**  *What type (hat, socks, t-shirt, sweater, etc.) of project is most popular?*

This question had to be adjusted due to the errors in data collection that was found. With the limitation of credits, time, and my free API service, I had to adjust my questions and insights to work with what I have. Overall, this should still give a general trend for designers to know the type of project category people favor.

In [6]:
# Top 5 pattern categories in the Ravelry dataset
# "Unknown" = patterns the API returned no category details for (specific to my dataset based on the fetch created)

# so drop duplicates to count each pattern once.
pattern_df = df.drop_duplicates(subset="pattern_id")

pattern_counts = (
    pattern_df[pattern_df["category"] != "Unknown"]["category"]
      .str.strip()
      .value_counts()
      .head(5)
      .reset_index()
)
pattern_counts.columns = ["category", "count"]

fig = px.bar(
    pattern_counts,
    x="category",
    y="count",
    title=f"Top 5 Pattern Categories Among Popular Patterns",
    labels={"category": "Category", "count": "Unique patterns"}
)
fig.show()

pattern_counts

,category,count
0,Pullover,117
1,Throw,69
2,Shawl / Wrap,55
3,Cardigan,48
4,Tee,37


**Interpretations — Question 1**

Among the 609 unique patterns in my sample, **Pullover is the most popular project type by a wide margin** (117 patterns, ~19%), followed by Throw (69), Shawl / Wrap (55), Cardigan (48), and Tee (37). Larger, wearable projects dominate: four of the top five categories are garments or wraps, which suggests that the patterns people actually start projects from skew toward bigger commitments rather than quick accessories.

**Concessions due to data collection errors.** This analysis is shaped by two issues found in the fetched data:

1. **The month filter in the fetch didn't work.** The `started=YYYY-MM` parameter on Ravelry's `/projects/search.json` endpoint did not actually filter projects by start month — every month returned the identical project list, so all 12 months (and therefore all 4 seasons) contain the same 611 pattern rows. My original seasonal version of this question had to be dropped; the chart above instead reports *overall* category popularity with duplicates removed (`drop_duplicates(subset="pattern_id")`), so each pattern is counted once rather than 12 times.
2. **Limited sample, not all of Ravelry.** With the free API's rate limits and the page cap in my fetch script, the sample is the ~600 patterns behind recently listed projects — a snapshot of currently popular patterns, not the full pattern database. One pattern with an "Unknown" category was excluded.

Within those limits, the ranking is still a useful directional signal for designers about which project categories crafters favor right now.

**Question 2:** *What weight and fiber of yarn is most popular by season?*

In [7]:
# Analysis for Question 2
# NOTE: the dataset's `season` column can't answer this — the fetch's month
# filter didn't work, so all four seasons contain the identical pattern set.
# Instead I derive season from each pattern's published_date (when the designer
# released it), which is real per-pattern data already in the dataset.

season_by_month = {12: "Winter", 1: "Winter", 2: "Winter",
                   3: "Spring", 4: "Spring", 5: "Spring",
                   6: "Summer", 7: "Summer", 8: "Summer",
                   9: "Fall", 10: "Fall", 11: "Fall"}
season_order = ["Winter", "Spring", "Summer", "Fall"]

# One row per pattern, with a publication season
pattern_df = df.drop_duplicates(subset="pattern_id").copy() # makes a new table 'pattern_df' with one row per unique pattern as a separate copy so I can safely add columns without warnings or side effects on df"
published = pd.to_datetime(pattern_df["published_date"], errors="coerce")
pattern_df["pub_season"] = published.dt.month.map(season_by_month)

# Drop rows where we can't answer the question (no publish date / unknown yarn info)
known = pattern_df.dropna(subset=["pub_season"])
known = known[(known["yarn_weight"] != "Unknown") & (known["fiber"] != "Unknown")]
print(f"Patterns with publish date + yarn info: {len(known)} of {len(pattern_df)}")

# The single most popular weight and fiber in each season
summary = pd.DataFrame({
    "most_popular_weight": known.groupby("pub_season")["yarn_weight"]
                                .agg(lambda s: s.value_counts().idxmax()),
    "most_popular_fiber": known.groupby("pub_season")["fiber"]
                               .agg(lambda s: s.value_counts().idxmax()),
    "patterns": known.groupby("pub_season").size(),
}).reindex(season_order)
print(summary)

# Full distributions: % of each season's patterns by weight and by fiber
# (percentages, not raw counts, because seasons have different numbers of patterns)
top_weights = known["yarn_weight"].value_counts().head(5).index
weight_pct = (pd.crosstab(known["pub_season"], known["yarn_weight"], normalize="index")
                .loc[season_order, top_weights] * 100).round(1)

fig = px.bar(
    weight_pct.reset_index().melt(id_vars="pub_season",
                                  var_name="yarn_weight", value_name="percent"),
    x="pub_season", y="percent", color="yarn_weight", barmode="group",
    title="Yarn Weight Mix by Publication Season (top 5 weights)",
    labels={"pub_season": "Season published", "percent": "% of season's patterns",
            "yarn_weight": "Yarn weight"},
)
fig.show()

top_fibers = known["fiber"].value_counts().head(5).index
fiber_pct = (pd.crosstab(known["pub_season"], known["fiber"], normalize="index")
               .loc[season_order, top_fibers] * 100).round(1)

fig = px.bar(
    fiber_pct.reset_index().melt(id_vars="pub_season",
                                 var_name="fiber", value_name="percent"),
    x="pub_season", y="percent", color="fiber", barmode="group",
    title="Fiber Mix by Publication Season (top 5 fibers)",
    labels={"pub_season": "Season published", "percent": "% of season's patterns",
            "fiber": "Fiber"},
)
fig.show()

Patterns with publish date + yarn info: 472 of 609
           most_popular_weight most_popular_fiber  patterns
pub_season                                                 
Winter                    Aran            Acrylic       116
Spring                      DK             Cotton       133
Summer                      DK             Merino       111
Fall                   Worsted             Merino       112


**Interpretations - Question 2**

Unlike the broken `season` column, publication season shows real variation — and it matches crafting intuition (n = 472 patterns with a publish date and known yarn info, out of 609):

- **Winter skews heavy and budget-friendly:** Aran is the top weight (25%) and Acrylic the top fiber (26%) — chunky, fast, affordable cold-weather projects.
- **Spring flips to light and plant-based:** DK takes over (27%) and Cotton peaks (22%), consistent with designers releasing warm-weather garments.
- **Summer stays light:** DK leads again and Fingering hits its high (~23%), though Merino is already the top fiber — likely designers publishing ahead for fall knitting.
- **Fall is wool season:** Worsted leads (22%) and Merino peaks at 29%, while Cotton drops to its low (11%).

The broad story for a designer or fiber vendor: yarn weight tracks temperature (heavier weights peak in Winter/Fall, lighter in Spring/Summer), and fiber follows the same cycle (plant fibers peak in Spring, wools in Fall/Winter, with Acrylic strongest in Winter).

**Caveats:** this measures when popular patterns were *published*, not when people knit them — designers often release a season ahead, which may explain Merino's summer strength. The differences are also modest (most shares move 5–15 points between seasons) and come from a sample of ~470 currently-popular patterns, so I'd treat these as directional trends rather than precise market shares.

**Question 3:** *Which designers are the most popular, and do they get there with a deep catalog or a single breakout hit?*

*Note: this question originally asked about designer experience (years since first Ravelry publish), but I dropped that angle — the experience value was derived from the earliest `published_date` among only the patterns in my sample, so it understates real tenure too much to be trustworthy.*

In [10]:
# Your analysis for Question 3
# Popularity = total favorites across the designer's patterns in this sample.
# (designer_years_experience is excluded: it was derived from only the patterns
# in this sample, so it's too unreliable to analyze.)

# One row per designer: dedupe patterns first, then keep each designer once
pattern_df = df.drop_duplicates(subset="pattern_id")
designers = (
    pattern_df.dropna(subset=["designer_id"])
              .drop_duplicates(subset="designer_id")
              [["designer_name", "designer_pattern_count",
                "designer_total_projects", "designer_total_favorites"]]
)
print(f"Unique designers in the sample: {len(designers)}")

# Top 10 most popular designers by total favorites
top10 = designers.nlargest(10, "designer_total_favorites").reset_index(drop=True)
print(top10)

fig = px.bar(
    top10.sort_values("designer_total_favorites"),
    x="designer_total_favorites",
    y="designer_name",
    orientation="h",
    color="designer_pattern_count",
    title="Top 10 Designers by Total Favorites (color = patterns in sample)",
    labels={"designer_total_favorites": "Total favorites (this sample)",
            "designer_name": "Designer",
            "designer_pattern_count": "Patterns in sample"},
)
fig.show()

# Catalog depth vs. popularity: do top designers need many patterns, or one hit?
fig = px.scatter(
    designers[designers["designer_total_favorites"] > 0],
    x="designer_pattern_count",
    y="designer_total_favorites",
    hover_name="designer_name",
    log_y=True,
    title="Catalog Depth vs. Popularity",
    labels={"designer_pattern_count": "Patterns in sample",
            "designer_total_favorites": "Total favorites (log scale)"},
)
fig.show()

print("Median patterns per designer, all designers:",
      designers["designer_pattern_count"].median())
print("Median patterns per designer, top 10:",
      top10["designer_pattern_count"].median())

Unique designers in the sample: 459
             designer_name  designer_pattern_count  designer_total_projects  \
0               PetiteKnit                    15.0                  31450.0   
1             Andrea Mowry                    10.0                  24266.0   
2  Ozetta : Hailey Smedley                    11.0                   9373.0   
3           Susanne Müller                     3.0                   9592.0   
4            Midori Hirose                     1.0                  36070.0   
5          Florence Miller                     2.0                  20458.0   
6              Camilla Vad                     1.0                   3870.0   
7            Orlane Sucche                     2.0                   6577.0   
8             Stephen West                     6.0                  29620.0   
9            Jane Crowfoot                     3.0                   5134.0   

   designer_total_favorites  
0                  398792.0  
1                  238041.0  
2   

Median patterns per designer, all designers: 1.0
Median patterns per designer, top 10: 3.0


**Interpretation:**

**PetiteKnit is in a league of her own** — roughly 399k total favorites across 15 patterns in the sample, nearly double the next designer (Andrea Mowry, ~238k). The top 10 is a mix of profiles: prolific names with many patterns in the sample (PetiteKnit, Ozetta, Andrea Mowry) sit alongside designers who got there on a single breakout hit (Midori Hirose and Camilla Vad each appear with just 1 pattern).

**A deep catalog is not required to reach the top.** Most designers in the sample appear with only one pattern (median = 1), and even within the top 10 the median is just a few patterns. The scatter plot shows highly favorited designers at every catalog size: some accumulate favorites across 10–15 trending patterns, while others match them with a single viral design. For an aspiring designer, that's encouraging — one strong pattern can break into the same tier as the most established names.

**Caveats:** favorites accumulate over a pattern's lifetime, so this ranking reflects all-time popularity of currently-trending patterns rather than popularity earned specifically in 2025–26. Every total is computed only over the ~600 patterns collected, so designers' real totals (and catalog sizes) on Ravelry are higher. However, this example helps to demonstrate the concept.

---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

In [11]:
# Visualization — supports my Question 3 finding:
# a single viral pattern can put a designer in the same tier as a deep catalog.

pattern_df = df.drop_duplicates(subset="pattern_id")

# The 10 most popular designers by total favorites
top10_names = (
    pattern_df.dropna(subset=["designer_id"])
              .drop_duplicates(subset="designer_id")
              .nlargest(10, "designer_total_favorites")["designer_name"]
)

# Each designer's bar is stacked from their individual patterns, so
# bar length = total popularity and number of segments = catalog depth.
top_patterns = pattern_df[pattern_df["designer_name"].isin(top10_names)]
designer_order = (
    top_patterns.groupby("designer_name")["favorites_count"]
                .sum().sort_values().index.tolist()
)

fig = px.bar(
    top_patterns,
    x="favorites_count",
    y="designer_name",
    orientation="h",
    hover_name="pattern_name",
    category_orders={"designer_name": designer_order},
    title="One viral pattern can rival a whole catalog:"
          "<br>how the top 10 designers earned their favorites",
    labels={"favorites_count": "Favorites (each segment = one pattern)",
            "designer_name": "Designer"},
)
# White outlines make the per-pattern segments visible within each bar
fig.update_traces(marker_line_color="white", marker_line_width=1.5)
fig.show()




**Chart rationale:**

**Why this chart type:** a horizontal stacked bar encodes both halves of my Question 3 finding in a single view. The total bar length compares overall popularity across the top 10 designers (a categorical comparison, so bars are appropriate), while the segments within each bar show *how* that popularity was earned — each segment is one pattern, sized by its favorites. Horizontal orientation keeps the designer names readable, and the white segment outlines make catalog depth visible at a glance. Hovering a segment reveals the pattern's name.

**What the reader should take away:** popularity at the top of Ravelry comes from two very different strategies. PetiteKnit's bar is long because it's built from many mid-sized segments (a deep catalog of consistently popular patterns), while designers like Midori Hirose and Camilla Vad reach the same tier with a bar that is essentially one giant segment — a single viral design. For an aspiring designer, the encouraging message is that a breakout hit can compete with years of catalog-building; for a yarn vendor, it means the trending-designer list can shift quickly when a single pattern goes viral.

**Summary of findings:**

My first two questions largely confirmed what I'd expect from my own experience as a knitter — pullovers dominate the popular-pattern categories, and yarn weight and fiber track the seasons (heavier wools and acrylics in fall/winter publications, lighter weights and cotton in spring/summer). The most important and surprising finding came from Question 3: there are two distinct paths to designer popularity, and a single viral pattern (like Midori Hirose's or Camilla Vad's) can rival the total favorites of a deep 15-pattern catalog like PetiteKnit's. What surprised me most about the process was how flawed my dataset turned out to be — the API's month filter silently failed, giving every season an identical pattern set, and it took several rounds of analysis (and skepticism toward AI-suggested fixes along the way) to uncover and work around it. With more time I would investigate the data collection pipeline itself: validating what the API actually returns before building an analysis on top of it, rather than discovering the flaws afterward. The main limitations are scope and sample size — ~600 currently-popular patterns with all-time favorite counts and publication dates as a season proxy — so these findings describe directional trends in what's trending now, not the full Ravelry market.

---

## Competency Claim

In a `mp1.md` file in your GitHub repository, write a short competency claim (2–4 sentences) for each domain you feel this project demonstrates. Be specific — cite something you actually did in this notebook.

Domains covered by this project typically include:
- **C3 — Data cleaning and file handling** (if you cleaned or reshaped data)
- **C5 — Data analysis with pandas** (answering questions with code)
- **C6 — Data visualization** (your chart)
- **C7 — Critical evaluation and professional judgment** (your interpretation and limitations section)

You don't have to claim every domain — only the ones your work actually demonstrates.